In [ ]:
from datetime import date, datetime

import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import numpy as np
import polars as pl
import spaceplot as sp

import tidal

sp.display('dark', retina=True)


In [ ]:
begin, end = '20260101', '20261231'
pred = tidal.noaa_tides(begin=begin, end=end, interval='5')
pred = tidal.normalize_amplitude(pred)


In [ ]:
dates = pred.unique(pl.col('t').dt.date())['t']
sun_times = tidal.get_sun_times(dates).sort('date')
sun_times.head()


In [ ]:
df = pred

day_tides = pl.DataFrame()
for daytime in ['sunrise', 'sunset']:
    time_tides = pl.DataFrame()
    for target in sun_times[daytime]:
        target = target.replace(tzinfo=None)
        res = df.with_columns(delta=(pl.col('t') - pl.lit(target)).abs()).sort('delta').head(1)

        time_tides = time_tides.vstack(res)

    time_tides = time_tides.with_columns(pl.lit(daytime).alias('daytime'))
    day_tides = day_tides.vstack(time_tides)

day_tides.head()


In [ ]:
axs = sp.montage_plot(1, panel_size=(18, 4))

sunrise_col = sp.plts.set25[2]
sunset_col = sp.plts.set25[1]

for daytime, col in zip(['sunrise', 'sunset'], [sunrise_col, sunset_col]):
    plot_df = day_tides.filter(pl.col('daytime') == daytime)
    axs.plot(plot_df['t'], plot_df['amplitude'], c=col, lw=0.2)
    axs.scatter(plot_df['t'], plot_df['amplitude'], color=col, s=1, label=daytime)

axs.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
axs.axvline(datetime.now(), color='teal', linestyle='-', lw=0.5)
axs.text(datetime.now(), 1.08, 'now  ', color='teal', fontsize=8, rotation=0, va='top', ha='right')

axs.legend(ncol=2, bbox_to_anchor=(1, 1), loc='upper right', borderaxespad=0, fancybox=False, framealpha=0.0)

axs.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
axs.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

axs.layout(
    y_lims=(-1.1, 1.1),
    x_margins=0,
    y_label='Tidal Amplitude',
    minor=False,
    x_tick_labelrotation=35,
    x_grid=True,
    grid_alpha=0.25,
)

for label in axs[0].get_xticklabels():
    label.set_horizontalalignment('right')
    label.set_rotation_mode('anchor')


In [ ]:
plot_df = pred.filter(pl.col('t').dt.date() == date.today())

In [ ]:
df = (
    day_tides.with_columns(pl.col('t').dt.month().alias('month'))
    .with_columns(pl.col('t').dt.week().alias('week'))
    .with_columns(pl.col('t').dt.weekday().alias('weekday'))
    .with_columns(pl.col('t').dt.quarter().alias('quarter'))
    .with_columns(pl.col('t').dt.strftime('%A').alias('weekday_name'))
    .with_columns(pl.col('t').dt.strftime('%B').alias('month_name'))
    .with_columns(pl.col('t').dt.day().alias('day'))
)

weekdays = df.unique('weekday_name').sort('weekday')['weekday_name'].to_list()
df_arrayed = (
    df.sort('weekday')
    .pivot(on='weekday_name', values='amplitude', index=['month', 'month_name', 'week', 'daytime'])
    .sort('month')
    .sort('week')
)


In [ ]:
import numpy as np
from astral import moon
from datetime import datetime, timedelta

d = datetime(2026, 1, 1)
end = datetime(2027, 1, 1)

FULL_MOON = 14.77

phases = []
while d < end:
    p = moon.phase(d)
    phases.append({'date': d, 'phase': p})
    d += timedelta(hours=1)


df = pl.DataFrame(phases)

target = FULL_MOON

t = df['date'].to_numpy()
p = df['phase'].to_numpy()

# Find intervals containing the target
idx = np.where((p[:-1] <= target) & (p[1:] >= target))[0]

dates = []
for i in idx:
    # linear interpolation
    frac = (target - p[i]) / (p[i + 1] - p[i])
    dt = t[i] + frac * (t[i + 1] - t[i])
    dates.append(dt)

dates


In [ ]:
df = moon_phases

(
    df.with_columns(
        phase_prev=pl.col('phase').shift(),
        date_prev=pl.col('date').shift(),
    )
    .filter((pl.col('phase_prev') < FULL_MOON) & (pl.col('phase') >= FULL_MOON))
    .with_columns(frac=((FULL_MOON - pl.col('phase_prev')) / (pl.col('phase') - pl.col('phase_prev'))))
    .with_columns(full_moon_date=(pl.col('date_prev') + ((pl.col('date') - pl.col('date_prev')) * pl.col('frac'))))
    # .select(full_moon_date, pl.lit(FULL_MOON).alias('phase'))
)


In [ ]:
axs = sp.montage_plot(1, panel_size=(8,2))
axs.plot(moon_phases['date'], moon_phases['phase'], color='gray', lw=0.5)

In [ ]:
plot_df = df.filter(pl.col('daytime') == 'sunset')

moon_df = sun_times.select('date', 'moon_phase_name').filter(pl.col('moon_phase_name').str.contains('Moon'))
moon_df = moon_df.unique([pl.col('date').dt.month(), 'moon_phase_name'])

axs = sp.montage_plot(1, panel_size=(18, 4))
radius = abs(plot_df['amplitude'] * 20)  # desired radius in points
norm = mcolors.Normalize(vmin=-1, vmax=1)
axs.scatter(
    plot_df['t'],
    np.ones(len(plot_df)),
    c=plot_df['amplitude'],
    s=radius**2,
    cmap='cmc.managua_r',
    label='sunset',
    norm=norm,
)

for row in moon_df.iter_rows(named=True):
    axs.text(
        row['date'],
        1.05,
        row['moon_phase_name'],
        fontsize=6,
        rotation=90,
        ha='center',
        va='bottom',
        color='gray',
    )

axs.layout(x_margins=0.01)


In [ ]:
sun_times.select('date', 'moon_phase_name')

In [ ]:
moon_df.filter(pl.col('moon_phase_name').str.contains('Full')).sort('date')

In [ ]:
# from cmcrameri import cm
# cm.show_cmaps()

In [ ]:
daytime = 'sunset'

axs = sp.montage_plot(
    12,
    n_cols=6,
    panel_size=(3, 3),
    layout='compressed',
    wspace=0.1,
    hspace=0.0,
    title=f'Tidal amplitude during {daytime.capitalize()}',
)

norm = mcolors.Normalize(vmin=-1, vmax=1)
for ax, mon in zip(axs, range(1, 13)):
    plot_df = df.filter(pl.col('month') == mon, pl.col('daytime') == daytime)

    radius = abs(plot_df['amplitude'] * 20)  # desired radius in points

    sc = ax.scatter(
        plot_df['weekday'],
        plot_df['week'],
        c=plot_df['amplitude'],
        norm=norm,
        s=radius**2,
        cmap='cmc.managua_r',
    )

    for row in plot_df.iter_rows(named=True):
        ax.text(row['weekday'], row['week'] + 0.5, row['day'], fontsize=8, ha='center', va='center', color='0.5')

    ax.set_title(f'{plot_df[0, "month_name"]}', fontsize=14, pad=8)

    times = (
        plot_df.group_by('week')
        .agg(pl.col('t').min())
        .with_columns(pl.col('t').dt.strftime('%H:%M').alias('date'))
        .sort('week')['date']
        .to_list()
    )

    sp.layout(
        ax,
        y_lims=(plot_df['week'].min() - 0.5, plot_df['week'].max() + 1),
        y_breaks=plot_df['week'].unique().to_list(),
        # y_ticklabels=times,
        y_tick_size=0,
    )

# after the plotting loop
cbar = axs.figure.colorbar(
    sc,
    ax=[axs[5], axs[11]],  # June and December
    location='right',
    shrink=0.5,
    fraction=1,
    pad=0.05,
)

axs.layout(x_ticklabels=[w[0:2] for w in weekdays], x_breaks=np.arange(1, len(weekdays) + 1))
sp.layout(axs[0:-6], x_ticks=False)
axs.invert_yaxis()

sp.show()


In [ ]:
axs = sp.montage_plot(1, panel_size=(18, 4))

axs.plot(pred['t'], pred['amplitude'], c='crimson', lw=0.2, label='Tidal amplitude')
axs.layout(
    y_lims=(-1.1, 1.1),
    x_margins=0,
    y_label='Tidal Amplitude',
    minor=False,
    # x_tick_labelrotation=35,
    x_grid=True,
    grid_alpha=0.25,
)


In [ ]:
axs = sp.montage_plot(1, panel_size=(8, 4))

days = 12

plot_df = pred.filter(pl.col('t').dt.date().is_between(date.today(), date.today() + timedelta(days=days)))

axs.plot(plot_df['t'], plot_df['amplitude'], c='crimson', lw=0.2, label='Tidal amplitude')
axs.layout(
    y_lims=(-1.1, 1.1),
    x_margins=0,
    y_label='Tidal Amplitude',
    minor=False,
    x_tick_labelrotation=35,
    x_grid=False,
    grid_alpha=0.25,
)

day = datetime.today()
day = day + timedelta(hours=6)

for i in range(days):
    axs.axvline(day + timedelta(days=i), color='royalblue', linestyle='--', lw=0.5, alpha=0.5)

axs.axvline(datetime.now(), color='teal', linestyle='-', lw=0.5)
# axs.text(datetime.now(), 1.08, 'now  ', color='teal', fontsize=8, rotation=0, va='top', ha='right')
axs.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)


In [ ]:
day